# Μάθημα 16 - Ανάπτυξη Κλιμακούμενων Πρακτόρων με το Microsoft Foundry

Σε αυτό το σημειωματάριο δημιουργείτε έναν **παραγωγικής χρήσης πράκτορα υποστήριξης πελατών** για την φανταστική εταιρεία **Contoso**. Σε αντίθεση με τα προηγούμενα μαθήματα, το θέμα εδώ δεν είναι ο βρόχος λογικής του πράκτορα — είναι όλα όσα τον περιβάλλουν και καθιστούν έναν πράκτορα ασφαλή για λειτουργία σε κλίμακα:

1. **Κλήση εργαλείων** — αναζητήσεις παραγγελιών και δημιουργία εισιτηρίων.
2. **RAG** — απαντήσεις πολιτικής από μια βάση γνώσεων.
3. **Μνήμη** — απομνημόνευση του πελάτη κατά τη διάρκεια των γύρων.
4. **Δρομολόγηση μοντέλων** — αποστολή απλών αιτημάτων σε μικρό μοντέλο, πολύπλοκων σε μεγάλο μοντέλο.
5. **Αποθήκευση προσωρινής απόκρισης** — εξυπηρέτηση επαναλαμβανόμενων ερωτήσεων χωρίς κλήση μοντέλου.
6. **Έγκριση ανθρώπου** — επιστροφές χρημάτων πάνω από ένα όριο σταματούν για έγκριση.
7. **Πύλη αξιολόγησης** — ένα σετ δοκιμών εκτός σύνδεσης που μπλοκάρει μια κακή κυκλοφορία.
8. **Παρατηρησιμότητα** — παρακολούθηση με OpenTelemetry γύρω από κάθε αίτημα.

Κάθε ενότητα είναι ανεξάρτητη και εκτελέσιμη. Διαβάστε κάθε γραμμή — οι παραγωγικές βασικές λειτουργίες είναι σκόπιμα μικρές.


## Εγκατάσταση

Πριν εκτελέσετε αυτό το σημειωματάριο, βεβαιωθείτε ότι έχετε:

1. **Ένα έργο Microsoft Foundry** με αναπτυγμένο μοντέλο συνομιλίας (π.χ. `gpt-4.1-mini`).
2. **Συνδεθεί με το Azure CLI** — εκτελέστε `az login` στο τερματικό σας.
3. **Ορίστε τις απαιτούμενες μεταβλητές περιβάλλοντος:**
   - `AZURE_AI_PROJECT_ENDPOINT` — το σημείο τερματισμού του έργου σας στο Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — το όνομα του αναπτυγμένου μοντέλου σας.

Η ενότητα RAG χρησιμοποιεί **Azure AI Search** όταν έχουν οριστεί οι `AZURE_SEARCH_SERVICE_ENDPOINT` και `AZURE_SEARCH_API_KEY`, και επιστρέφει σε αναζήτηση στη μνήμη ώστε το σημειωματάριο να εκτελείται χωρίς πόρο Αναζήτησης.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Εργαλεία

Τα εργαλεία παραγωγής εκτελούν πραγματική δουλειά σε πραγματικά συστήματα. Εδώ προσομοιώνουμε μια βάση δεδομένων παραγγελιών και ένα σύστημα έκδοσης εισιτηρίων με απλές συναρτήσεις Python. Ο διακοσμητής `@tool` τα εκθέτει στον πράκτορα.

Σημειώστε ότι το `issue_refund` χρησιμοποιεί `approval_mode="always_require"` για επιστροφές πάνω από ένα όριο — αυτή είναι η πρωτόγονη ανθρώπινη παρέμβαση που εφαρμόζουμε αργότερα.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Βάση Γνώσης Πολιτικής

Οι ερωτήσεις πολιτικής ("ποιο είναι το παράθυρο επιστροφής σας;") θα πρέπει να απαντώνται από μια αξιόπιστη πηγή και όχι από τη μνήμη του μοντέλου. Τυλίγουμε μια μικρή βάση γνώσης ως εργαλείο αναζήτησης.

Στην παραγωγή αυτό είναι **Azure AI Search**· εδώ παρέχουμε μια αναζήτηση με λέξεις-κλειδιά εντός μνήμης ώστε το τετράδιο να εκτελείται οπουδήποτε, και αλλάζουμε αυτόματα σε Azure AI Search όταν υπάρχουν οι μεταβλητές περιβάλλοντος.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Μνήμη

Ένας εκπρόσωπος υποστήριξης που ξεχνά σε ποιον μιλάει είναι κακός εκπρόσωπος υποστήριξης. Κρατάμε μια μικρή αποθήκη προφίλ ανά πελάτη και εισάγουμε μια σύντομη περίληψη στις οδηγίες του εκπροσώπου. Στην παραγωγή αυτή είναι μια υπηρεσία μνήμης (βλ. Μάθημα 13)· εδώ ένα dict κάνει το μοτίβο ορατό.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Δρομολόγηση Μοντέλου και Αποθήκευση Απαντήσεων

Δύο μοχλοί κόστους συνδεδεμένοι σε έναν και μόνο χειριστή αιτήσεων:

- **Δρομολόγηση**: ένας φθηνός ευρετικός ταξινομητής αποφασίζει αν ένα αίτημα χρειάζεται το μικρό ή το μεγάλο μοντέλο.
- **Αποθήκευση**: κανονικοποιημένες επαναλαμβανόμενες ερωτήσεις εξυπηρετούνται απευθείας από μια cache χωρίς κλήση μοντέλου.

Ο ταξινομητής εδώ είναι σκόπιμα απλός. Στην παραγωγή θα τον επικυρώσετε έναντι της κίνησης και θα μπορούσατε να τον αντικαταστήσετε με τον Model Router του Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Ο Πράκτορας, η Ανθρώπινη Έγκριση και η Παρατηρησιμότητα

Τώρα συναρμολογούμε τον πράκτορα από τα παραπάνω εργαλεία και τυλίγουμε κάθε αίτημα σε ένα span του OpenTelemetry. Η συνάρτηση `handle_support_request` είναι ο handler αιτημάτων παραγωγής: cache → route → trace → run → cache.

Η ανθρώπινη έγκριση διαχειρίζεται από το πλαίσιο: επειδή το `issue_refund` έχει `approval_mode="always_require"`, η εκτέλεση παίρνει παύση και εμφανίζει ένα αίτημα έγκρισης που λύνουμε ρητά.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Πύλη Αξιολόγησης

Αυτή είναι η πύλη κυκλοφορίας από το μάθημα: ένα σετ δοκιμών εκτός σύνδεσης βαθμολογεί τον πράκτορα, και η ανάπτυξη προχωρά μόνο αν το ποσοστό επιτυχίας ξεπεράσει ένα όριο. Ο βαθμολογητής εδώ είναι ένας απλός έλεγχος επικαλύψεων λέξεων-κλειδιών για να διατηρείται το σημειωματάριο αυτόνομο· σε παραγωγή θα χρησιμοποιούσατε έναν LLM-ως-κριτή ή έναν αξιολογητή πλαισίου (βλέπε Μάθημα 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Εκτέλεση σε Συνδυασμό: Μια Προσομοιωμένη Έκδοση

Το κελί παρακάτω δείχνει ολόκληρο το βρόχο που περιγράφει το μάθημα: εκτέλεση της πύλης αξιολόγησης και μόνο «ανάπτυξη» εάν αυτή περάσει. Αυτό είναι το πρότυπο που θα εκτελούσατε στο CI πριν προωθήσετε μια έκδοση πράκτορα στην Υπηρεσία Πράκτορα Foundry.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Περίληψη

Συναρμολογήσατε έναν πράκτορα υποστήριξης πελατών έτοιμο για παραγωγή με κάθε λειτουργικό ζήτημα καλυμμένο:

- **Εργαλεία, RAG και μνήμη** δίνουν τη δυνατότητα και το πλαίσιο στον πράκτορα.
- **Δρομολόγηση μοντέλων και caching** διατηρούν τον χρόνο απόκρισης και το κόστος υπό έλεγχο.
- **Εγκρίσεις από ανθρώπους** προστατεύουν κινήσεις υψηλού κινδύνου όπως μεγάλες επιστροφές χρημάτων.
- **Η πύλη αξιολόγησης** μπλοκάρει κακές κυκλοφορίες πριν αποσταλούν.
- **Ιχνηλάτηση** καθιστά κάθε αίτημα παρατηρήσιμο.

### Πρόκληση

Επεκτείνετε αυτόν τον πράκτορα για να:

1. **Υποστηρίξετε πολλαπλά μοντέλα** — προσθέστε ένα τρίτο επίπεδο "λογικής" και δρομολογήστε τις κλιμακώσεις/παραπονούμενες σε αυτό.
2. **Προσθέσετε πύλες αξιολόγησης** — επεκτείνετε το `TEST_CASES` για να περιλαμβάνει σενάρια έγκρισης επιστροφών και επιβεβαιώστε ότι η πύλη εντοπίζει οπισθοδρομήσεις.
3. **Προσθέσετε δρομολόγηση με γνώμονα το κόστος** — παρακολουθήστε ένα εκτιμώμενο κόστος ανά αίτημα (μικρό έναντι μεγάλου έναντι cache) και εκτυπώστε μια αναφορά κόστους μετά από παρτίδα μεικτών ερωτήσεων.

Στο επόμενο μάθημα ακολουθείτε το αντίθετο μονοπάτι και τρέχετε έναν πράκτορα ολοκληρωτικά στο δικό σας μηχάνημα με το Microsoft Foundry Local και το Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
